In [2]:
!wget -O blast.py https://raw.githubusercontent.com/spcl/blast/refs/heads/main/src/spmm.py

--2026-04-20 16:13:47--  https://raw.githubusercontent.com/spcl/blast/refs/heads/main/src/spmm.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18317 (18K) [text/plain]
Saving to: ‘blast.py’

blast.py            100%[===================>]  17.89K  --.-KB/s    in 0.001s  

2026-04-20 16:13:47 (13.9 MB/s) - ‘blast.py’ saved [18317/18317]



In [3]:
import csv
import dataclasses
import json
import os
from pathlib import Path

import numpy as np
import torch
import triton
from blast import BCSC_SpMM, from_block_sparsity_mask

torch.manual_seed(42)

print(f"GPU: {torch.cuda.get_device_name(0)}\n")

GPU: Tesla T4



/content/blast.py:145: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(sparsity_mask.sum() == block_size * block_size * block_sparsity_mask.sum(), "Number of nonzeros in sparsity mask and block sparsity mask do not match")


In [4]:
@dataclasses.dataclass
class Config:
    M: int  # number of rows in X
    K: int  # number of columns in X and rows in W
    blk_size: int
    sparsity: float
    # "controls the number of rows from the dense matrix that participate in
    # the thread-block level reduction, thus increasing the reuse of data
    # loaded into shared memory from the sparse block" (see page 4 of BLaST)
    # they used 128, which was really slow on Colab. 32 gave best performance
    # out of 32, 64, and 128 on Colab's T4 GPU
    blk_M: int = 32

    def __post_init__(self):
        # number of columns in W. The paper always uses 4 * K
        self.N = 4 * self.K
        self.K_blocks = self.K // self.blk_size
        self.N_blocks = self.N // self.blk_size
        self.total_blocks = self.K_blocks * self.N_blocks
        density = 1 - self.sparsity
        self.num_nonzero_blocks = max(1, int(self.total_blocks * density))

    def write_to(self, path: Path):
        with path.open("w") as f:
            f.write(f"M={self.M} K={self.K} N={self.N}\n")
            f.write(f"blk_size={self.blk_size}\n")
            f.write(f"sparsity={self.sparsity}\n")
            f.write(f"num_nonzero_blocks={self.num_nonzero_blocks}\n")

    def calculate_tflops(self, ms: float) -> float:
        density = 1 - self.sparsity
        return 2 * self.M * self.K * self.N * density / (ms / 1000.0) / 1e12

In [5]:
def generate_sparse_matrix(config: Config) -> torch.Tensor:
    total_blocks, num_nonzero = config.total_blocks, config.num_nonzero_blocks
    flat_mask = torch.zeros(total_blocks, dtype=torch.bool, device="cuda")
    flat_mask[torch.randperm(total_blocks, device="cuda")[:num_nonzero]] = True
    block_mask = flat_mask.view(config.K_blocks, config.N_blocks)

    W_bcsc = from_block_sparsity_mask(
        block_mask, block_size=config.blk_size, dtype=torch.float16
    )
    # fill with random data since the values returned are all zero
    W_bcsc.values().normal_()
    return W_bcsc


def save_arr(arr: torch.Tensor, path: Path, dtype: torch.dtype | None = None):
    arr = arr.cpu().numpy()
    if dtype is not None:
        arr = arr.astype(dtype)
    arr.tofile(path)

In [6]:
matrices_dir = Path("./matrices").resolve()
matrices_dir.mkdir(exist_ok=True)
results_csv = Path("./blast_results.csv").resolve()

In [7]:
# base sparsity doesn't matter here since it gets replaced; 0.50 is just temp
base_config = Config(M=1024, K=4096, blk_size=64, sparsity=0.50)
sparsities = [0.50, 0.70, 0.80, 0.90, 0.95]
warmup_iters = 10
n_iters = 50

# (sparsity, median ms, median tflops)
results: list[tuple[float, float, float]] = []

for sparsity in sparsities:
    print(f"--- Sparsity {sparsity*100:.0f}% ---")

    config = dataclasses.replace(base_config, sparsity=sparsity)
    W_bcsc = generate_sparse_matrix(config)
    # The 1 is the batch dimension (since the intended purpose of BLaST is
    # machine learning stuff)
    X = torch.randn(1, config.M, config.K, device="cuda", dtype=torch.float16)

    blk_M, N = config.blk_M, config.N
    for _ in range(warmup_iters):
        BCSC_SpMM(X, W_bcsc, blk_M=blk_M, N=N, batch_dim_first=True)

    starts = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]
    ends = [torch.cuda.Event(enable_timing=True) for _ in range(n_iters)]

    for i in range(n_iters):
        starts[i].record()
        BCSC_SpMM(X, W_bcsc, blk_M=blk_M, N=N, batch_dim_first=True)
        ends[i].record()

    torch.cuda.synchronize()
    times = [s.elapsed_time(e) for s, e in zip(starts, ends)]
    median_ms = round(np.median(times), 4)
    tflops = config.calculate_tflops(median_ms)
    print(f"  {median_ms:.3f} ms  |  {tflops:.2f} TFLOP/s")
    results.append((sparsity, median_ms, tflops))

    out_dir = matrices_dir / f"sp{int(sparsity*100)}"
    out_dir.mkdir(exist_ok=True)

    config.write_to(out_dir / "config.txt")
    # squeeze(0) gets rid of the batch dimension
    save_arr(X.squeeze(0), out_dir / "X.bin")
    save_arr(W_bcsc.values(), out_dir / "W_vals.bin")
    # torch uses int64 for the indices but we only need int32
    save_arr(W_bcsc.row_indices(), out_dir / "W_row_idx.bin", np.int32)
    save_arr(W_bcsc.ccol_indices(), out_dir / "W_col_ptr.bin", np.int32)
    save_arr(W_bcsc.to_dense(), out_dir / "W_dense.bin")

    del W_bcsc, X
    torch.cuda.empty_cache()

with results_csv.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["sparsity", "median_ms", "median_tflops"])
    writer.writerows(results)

print(f"\nResults saved to {results_csv}")
print(f"Matrices saved to {matrices_dir}/")

--- Sparsity 50% ---


/content/blast.py:35: UserWarning: Sparse BSC tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  return torch.sparse_bsc_tensor(colPtr, rowIdx, values, dtype=dtype)


  56.343 ms  |  1.22 TFLOP/s
--- Sparsity 70% ---
  34.851 ms  |  1.18 TFLOP/s
--- Sparsity 80% ---
  24.114 ms  |  1.14 TFLOP/s
--- Sparsity 90% ---
  12.905 ms  |  1.07 TFLOP/s
--- Sparsity 95% ---
  7.391 ms  |  0.93 TFLOP/s

Results saved to /content/blast_results.csv
Matrices saved to /content/matrices/


In [8]:
%%writefile benchmark.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_fp16.h>
#include <cublas_v2.h>
#include <algorithm>
#include <cuda_runtime.h>

#define CHECK_CUDA(call) { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error in %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
}

#define CHECK_CUBLAS(call) { \
    cublasStatus_t err = call; \
    if (err != CUBLAS_STATUS_SUCCESS) { \
        fprintf(stderr, "cuBLAS error in %s:%d: %d\n", __FILE__, __LINE__, (int)err); \
        exit(EXIT_FAILURE); \
    } \
}

void* read_bin_file(const char* dir, const char* fname, size_t num_bytes) {
  char path[512];
  snprintf(path, sizeof(path), "%s/%s", dir, fname);
  FILE* f = fopen(path, "rb");
  if (!f) { fprintf(stderr, "Cannot open %s\n", path); exit(1); }
  void* buf = malloc(num_bytes);
  fread(buf, 1, num_bytes, f);
  fclose(f);
  return buf;
}

struct Config {
    int M, K, N, blk_size, num_nnz;
    float sparsity;
};

static Config parse_config(const char *dir) {
    char path[256];
    snprintf(path, sizeof(path), "%s/config.txt", dir);
    FILE *f = fopen(path, "r");
    if (!f) {
      fprintf(stderr, "Cannot open %s\n", path);
      exit(1);
    }

    Config c = {};
    char line[256];
    while (fgets(line, sizeof(line), f)) {
        sscanf(line, "M=%d K=%d N=%d", &c.M, &c.K, &c.N);
        sscanf(line, "blk_size=%d", &c.blk_size);
        sscanf(line, "sparsity=%f", &c.sparsity);
        sscanf(line, "num_nonzero_blocks=%d", &c.num_nnz);
    }
    fclose(f);
    printf("Config: M=%d K=%d N=%d blk=%d sparsity=%.2f nnz_blocks=%d\n",
           c.M, c.K, c.N, c.blk_size, c.sparsity, c.num_nnz);
    return c;
}

void benchmark(int sparsity) {
  printf("Sparsity: %d\n", sparsity);
  char dir[128];
  snprintf(dir, sizeof(dir), "/content/matrices/sp%d", sparsity);

  Config cfg = parse_config(dir);
  int M = cfg.M;
  int K = cfg.K;
  int N = cfg.N;
  int blk = cfg.blk_size;
  int num_nnz = cfg.num_nnz;

  size_t X_bytes = (size_t)M * K * sizeof(half);
  half* h_X = (half*)read_bin_file(dir, "X.bin", X_bytes);

  size_t W_bytes = (size_t)K * N * sizeof(half);
  half* h_W = (half*)read_bin_file(dir, "W_dense.bin", W_bytes);

  size_t vals_bytes = (size_t)num_nnz * blk * blk * sizeof(half);
  half* h_vals = (half*)read_bin_file(dir, "W_vals.bin", vals_bytes);

  size_t row_idx_bytes = (size_t)num_nnz * sizeof(int);
  int* h_row_idx = (int*) read_bin_file(dir, "W_row_idx.bin", row_idx_bytes);

  size_t col_ptr_bytes = (size_t)(N / blk + 1) * sizeof(int);
  int* h_col_ptr = (int*) read_bin_file(dir, "W_col_ptr.bin", col_ptr_bytes);

  // copying to GPU here...

  half *d_X, *d_W, *d_Y;

  size_t Y_bytes = (size_t)M * N * sizeof(half);

  CHECK_CUDA(cudaMalloc(&d_X, X_bytes));
  CHECK_CUDA(cudaMalloc(&d_W, W_bytes));
  CHECK_CUDA(cudaMalloc(&d_Y, Y_bytes));

  CHECK_CUDA(cudaMemcpy(d_X, h_X, X_bytes, cudaMemcpyHostToDevice));
  CHECK_CUDA(cudaMemcpy(d_W, h_W, W_bytes, cudaMemcpyHostToDevice));

  cublasHandle_t handle;
  CHECK_CUBLAS(cublasCreate(&handle));

  half alpha = 1.0;
  half beta = 0.0;
  int n_iters = 50;

  // benchmark stuff here...

  for(int i = 0; i < 10; i++){
    CHECK_CUBLAS(cublasHgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, d_W, N, d_X, K, &beta, d_Y, N));
  }


  CHECK_CUDA(cudaDeviceSynchronize());

  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  for(int i = 0; i < n_iters; i++){
    CHECK_CUBLAS(cublasHgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, M, K, &alpha, d_W, N, d_X, K, &beta, d_Y, N));
  }

  cudaEventRecord(stop);

  cudaEventSynchronize(stop);

  float milliseconds = 0;
  cudaEventElapsedTime(&milliseconds, start, stop);

  CHECK_CUDA(cudaDeviceSynchronize());

  float avg_milliseconds = milliseconds/n_iters;
  printf("Average cuBLAS Math time: %f ms\n", avg_milliseconds);


  // freeing GPU stuff here...

  free(h_X);
  free(h_W);
  free(h_vals);
  free(h_row_idx);
  free(h_col_ptr);
  CHECK_CUBLAS(cublasDestroy(handle));
  CHECK_CUDA(cudaFree(d_X));
  CHECK_CUDA(cudaFree(d_W));
  CHECK_CUDA(cudaFree(d_Y));
}

int main() {
  // sparsities = [0.50, 0.70, 0.80, 0.90, 0.95]
  int sparsities[] = {50, 70, 80, 90, 95};
  for (int i = 0; i < 5; i++) {
    benchmark(sparsities[i]);
  }
}

Overwriting benchmark.cu


In [9]:
!nvcc benchmark.cu -o benchmark -arch=sm_75 -lcublas
!./benchmark

Sparsity: 50
Config: M=1024 K=4096 N=16384 blk=64 sparsity=0.50 nnz_blocks=8192
Average cuBLAS Math time: 4.981593 ms
Sparsity: 70
Config: M=1024 K=4096 N=16384 blk=64 sparsity=0.70 nnz_blocks=4915
Average cuBLAS Math time: 4.798915 ms
Sparsity: 80
Config: M=1024 K=4096 N=16384 blk=64 sparsity=0.80 nnz_blocks=3276
Average cuBLAS Math time: 4.761860 ms
Sparsity: 90
Config: M=1024 K=4096 N=16384 blk=64 sparsity=0.90 nnz_blocks=1638
Average cuBLAS Math time: 4.570563 ms
Sparsity: 95
Config: M=1024 K=4096 N=16384 blk=64 sparsity=0.95 nnz_blocks=819
Average cuBLAS Math time: 4.526256 ms
